In [1]:
!pip install opencv-python numpy pandas mediapipe

  Using cached absl_py-2.1.0-py3-none-any.whl.metadata (2.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 12.4 MB/s eta 0:00:0000:0100:01
Using cached absl_py-2.1.0-py3-none-any.whl (133 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.0/102.0 MB 12.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 13.0 MB/s eta 0:00:00a 0:00:01


In [ ]:
import cv2
import mediapipe as mp
import pandas as pd
import os

mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# Initialize Pose with high accuracy model.
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Define the columns we want (only x and y)
LANDMARK_COLUMNS = [
    'frame',
    'left_wrist_x', 'left_wrist_y',
    'left_index_x', 'left_index_y',
    'left_pinky_x', 'left_pinky_y',
    'left_thumb_x', 'left_thumb_y',
    'right_wrist_x', 'right_wrist_y',
    'right_index_x', 'right_index_y',
    'right_pinky_x', 'right_pinky_y',
    'right_thumb_x', 'right_thumb_y'
]

def get_filename_without_extension(path: str) -> str:
    """
    Utility function to extract the filename without its extension.
    For example, 'C:/Videos/sample.mp4' -> 'sample'
    """
    return os.path.splitext(os.path.basename(path))[0]

def process_single_video(video_path: str) -> pd.DataFrame:
    """
    Process a single video to extract specified pose landmark coordinates (x and y only).
    Returns a pandas DataFrame with the columns defined in LANDMARK_COLUMNS.
    """
    cap = cv2.VideoCapture(video_path)

    frame_count = 0
    all_rows = []
    last_seen = {}  # Keep track of the last seen valid coordinate for each landmark

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert the frame to RGB before processing
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)

        # Initialize the row dictionary for the current frame
        row = {'frame': frame_count}

        # If pose landmarks are detected, extract the x and y coordinates
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            for lr in ['left', 'right']:
                for lm in ['wrist', 'index', 'pinky', 'thumb']:
                    for axis in ['x', 'y']:
                        key = f'{lr}_{lm}_{axis}'
                        # Get the official Mediapipe landmark index
                        landmark_index = getattr(mp_pose.PoseLandmark, f'{lr.upper()}_{lm.upper()}', None)
                        # If the index is valid, and the total landmarks are sufficient
                        if landmark_index is not None and len(landmarks) > landmark_index:
                            landmark = landmarks[landmark_index]
                            value = getattr(landmark, axis, None)
                            if value is not None:
                                row[key] = value
                                last_seen[key] = value
                            elif key in last_seen:
                                # Use the last seen value if the current frame doesn't have it
                                row[key] = last_seen[key]
                        elif key in last_seen:
                            # Use the last seen value if the landmark is not valid for this frame
                            row[key] = last_seen[key]

        all_rows.append(row)
        frame_count += 1

    cap.release()
    cv2.destroyAllWindows()

    # Convert all_rows to a DataFrame
    df = pd.DataFrame(all_rows, columns=LANDMARK_COLUMNS)
    return df

def save_to_csv(df: pd.DataFrame, output_dir: str, video_name: str):
    """
    Saves the DataFrame to a CSV file in the specified output directory.
    The file is named after the video_name with '_landmarks_coordinates.csv' appended.
    """
    os.makedirs(output_dir, exist_ok=True)  # Create the directory if it doesn't exist
    csv_file_path = os.path.join(output_dir, f'{video_name}_landmarks_coordinates.csv')
    df.to_csv(csv_file_path, index=False)
    print(f"Data saved to {csv_file_path}")

def process_video_directory(input_directory: str, output_directory: str):
    """
    Processes each .mp4 video file found in the input_directory:
      1. Extract pose landmarks (x and y) for each video.
      2. Saves each set of landmarks to a CSV in the output_directory.
    """
    # Iterate over all files in input_directory
    for filename in os.listdir(input_directory):
        if filename.lower().endswith('.mp4'):
            video_path = os.path.join(input_directory, filename)
            video_name = get_filename_without_extension(video_path)

            print(f"Processing video: {video_path}")
            df_landmarks = process_single_video(video_path)
            save_to_csv(df_landmarks, output_directory, video_name)

    print("All videos in the directory have been processed.")


In [ ]:
# ===============================
# Example Usage (uncomment below to run):
# ===============================

# input_dir = 'Processed_Videos/Experts'
# output_dir = 'CSV_Files/Experts/Landmark_CSVs'

# process_video_directory(input_dir, output_dir)

In [3]:
input_dir = 'Processed_Videos/Novices'
output_dir = 'CSV_Files/Novices/Landmark_CSVs'

process_video_directory(input_dir, output_dir)

Processing video: Processed_Videos/Novices\15_task1.mp4


c:\Users\Admin\anaconda3\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Data saved to CSV_Files/Novices/Landmark_CSVs\15_task1_landmarks_coordinates.csv
Processing video: Processed_Videos/Novices\16_task1.mp4
Data saved to CSV_Files/Novices/Landmark_CSVs\16_task1_landmarks_coordinates.csv
Processing video: Processed_Videos/Novices\17_task1.mp4
Data saved to CSV_Files/Novices/Landmark_CSVs\17_task1_landmarks_coordinates.csv
Processing video: Processed_Videos/Novices\20_task1.mp4
Data saved to CSV_Files/Novices/Landmark_CSVs\20_task1_landmarks_coordinates.csv
All videos in the directory have been processed.


In [ ]:
# If you only want to process a single video:
# single_video_path = 'Cropped_Videos/Processed_Videos/Experts/107_gearbox.mp4'
# single_video_df = process_single_video(single_video_path)
# video_name = get_filename_without_extension(single_video_path)
# save_to_csv(single_video_df, 'CSV_Files/Experts/Landmark_CSVs', video_name)